# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranked queue prioritizes content using low CTR, content staleness, and the model score. Higher-scoring items should be reviewed first.

Reason codes:

LOW_CTR — Content has low click-through rate.
STALE_CONTENT — Content has not been updated recently.
MODEL_PRIORITY — The model identified a higher likelihood of a downward trend.

The recommendations are decision-support and require human review.

In [1]:
import pandas as pd
import numpy as np
import os

url = "https://raw.githubusercontent.com/usmanumer038/ml-internship-work/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df = df.dropna(subset=[
    "ctr",
    "impressions_90d",
    "days_since_last_update"
]).copy()

df = df[df["impressions_90d"] >= 100]

# Create priority score
df["priority_score"] = (
    0.6 * df["ctr"].rank(pct=True, ascending=True) +
    0.4 * df["days_since_last_update"].rank(pct=True)
)

# Reason codes
df["reason_code"] = np.where(
    df["ctr"] < 0.05,
    "LOW_CTR",
    "STALE_CONTENT"
)

# Recommended action
df["action"] = np.where(
    df["reason_code"] == "LOW_CTR",
    "REVIEW_TITLE_META",
    "REFRESH_CONTENT"
)

queue = df.sort_values(
    "priority_score",
    ascending=False
).copy()

queue["rank"] = range(1, len(queue) + 1)

display(
    queue[
        ["rank", "content_id", "priority_score",
         "reason_code", "action"]
    ].head(20)
)

,rank,content_id,priority_score,reason_code,action
1735,1,content_c6a9f1c16dee,0.999773,STALE_CONTENT,REFRESH_CONTENT
6233,2,content_ba00ffc6318c,0.999364,STALE_CONTENT,REFRESH_CONTENT
191,3,content_4729b57ca036,0.998918,STALE_CONTENT,REFRESH_CONTENT
665,4,content_e444c00065bd,0.998914,STALE_CONTENT,REFRESH_CONTENT
8006,5,content_cb7e312f5d32,0.997405,STALE_CONTENT,REFRESH_CONTENT
9158,6,content_a001aa4be7b7,0.996974,STALE_CONTENT,REFRESH_CONTENT
12400,7,content_482aff19e9cc,0.996592,STALE_CONTENT,REFRESH_CONTENT
40,8,content_4df0b7207fe3,0.993070,STALE_CONTENT,REFRESH_CONTENT
16648,9,content_69fad7e6c50c,0.986545,STALE_CONTENT,REFRESH_CONTENT
13070,10,content_c8f7e3b9891c,0.976184,STALE_CONTENT,REFRESH_CONTENT


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use**

This playbook is designed to help a content or SEO reviewer identify which content should be reviewed first. The ranked queue supports prioritization and does not automatically make changes.

**Limits**

The recommendations are based on available performance signals and simple scoring logic. A high score does not guarantee that content needs to be changed. Search intent, business priorities, and recent changes may affect the correct decision.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review**

Before taking action, a reviewer should check:

Search intent
Content quality
Recent updates
Business importance
Whether low CTR is expected for the topic

**What Should NOT Be Automated**

The system should not automatically:

Publish or rewrite content
Change titles or meta descriptions
Delete content
Make business decisions

All recommendations require human review before action.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The playbook should be reviewed or updated when:

Content performance patterns change significantly.
CTR distributions change.
The model performance decreases.
New data becomes available.
Recommendations are frequently rejected by human reviewers.

These are monitoring signals rather than automatic retraining instructions.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked queue is exported to work/outputs/ so it can be reused in the research paper.

In [3]:
import os

os.makedirs("work/outputs", exist_ok=True)

output = queue[
    ["rank", "content_id", "priority_score",
     "reason_code", "action",
     "ctr", "impressions_90d",
     "days_since_last_update"]
]

output.to_csv(
    "work/outputs/content_action_playbook.csv",
    index=False
)

print("Queue exported successfully.")
print("Rows:", len(output))

Queue exported successfully.
Rows: 22006


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.